# Adaptive RAG agent with LangGraph and NVIDIA Nemotron

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-nvidia/blob/main/cookbook/langgraph_rag_agent_nvidia_nim.ipynb)

This notebook builds a RAG agent with [LangGraph](https://docs.langchain.com/oss/python/langgraph/overview)
that combines ideas from three RAG papers:

- **Routing:** Adaptive RAG ([paper](https://arxiv.org/abs/2403.14403)). Route questions to different retrieval approaches.
- **Fallback:** Corrective RAG ([paper](https://arxiv.org/abs/2401.15884)). Fall back to web search if the retrieved documents are not relevant.
- **Self-correction:** Self-RAG ([paper](https://arxiv.org/abs/2310.11511)). Retry answers that are not grounded in the documents or do not address the question.

## NVIDIA models

- [`ChatNVIDIA`](https://docs.langchain.com/oss/python/integrations/chat/nvidia_ai_endpoints) with `nvidia/nemotron-3-super-120b-a12b`
- [`NVIDIAEmbeddings`](https://docs.langchain.com/oss/python/integrations/text_embedding/nvidia_ai_endpoints) with `nvidia/nemotron-3-embed-1b`
- `NVIDIARerank` with `nvidia/llama-nemotron-rerank-vl-1b-v2`

All three run on NVIDIA-hosted endpoints or on your own [NVIDIA NIM](https://developer.nvidia.com/nim) containers.

## Install the packages

In [ ]:
%pip install -qU "langchain-nvidia-ai-endpoints>=1.4,<2" "langchain>=1.0,<2" "langgraph>=1.0,<2" "langchain-community>=0.4,<1" "langchain-text-splitters>=1.0,<2" "langchain-tavily>=0.2,<1" beautifulsoup4

## Configure the endpoint: hosted or self-hosted

Every model client in this notebook is built from the next cell.

| Variable | Unset (default) | Set |
| --- | --- | --- |
| `NVIDIA_BASE_URL` | NVIDIA-hosted endpoints on [build.nvidia.com](https://build.nvidia.com/) | Your own NIM, e.g. `http://localhost:8000/v1` |
| `NVIDIA_API_KEY` | Required for hosted endpoints (you are prompted if it is missing) | Optional for most self-hosted NIMs |
| `NVIDIA_CHAT_MODEL` | `nvidia/nemotron-3-super-120b-a12b` | The model your NIM serves (see `GET $NVIDIA_BASE_URL/models`) |
| `NVIDIA_EMBED_BASE_URL` / `NVIDIA_RERANK_BASE_URL` | Same as `NVIDIA_BASE_URL` | Separate embedding / reranking NIMs, e.g. `http://localhost:8001/v1` |
| `NVIDIA_EMBED_MODEL` / `NVIDIA_RERANK_MODEL` | `nvidia/nemotron-3-embed-1b` / `nvidia/llama-nemotron-rerank-vl-1b-v2` | The models your NIMs serve |
| `TAVILY_API_KEY` | Web search is replaced by an offline stub | [Tavily](https://tavily.com/) web search is enabled |

In [ ]:
import getpass
import os

# Unset: NVIDIA-hosted endpoints (build.nvidia.com).
# Set, e.g. "http://localhost:8000/v1": a self-hosted NIM.
NVIDIA_BASE_URL = os.environ.get("NVIDIA_BASE_URL") or None
# Self-hosted embedding and reranking NIMs usually run as separate containers.
EMBED_BASE_URL = os.environ.get("NVIDIA_EMBED_BASE_URL") or NVIDIA_BASE_URL
RERANK_BASE_URL = os.environ.get("NVIDIA_RERANK_BASE_URL") or NVIDIA_BASE_URL

CHAT_MODEL = os.environ.get("NVIDIA_CHAT_MODEL", "nvidia/nemotron-3-super-120b-a12b")
EMBED_MODEL = os.environ.get("NVIDIA_EMBED_MODEL", "nvidia/nemotron-3-embed-1b")
RERANK_MODEL = os.environ.get("NVIDIA_RERANK_MODEL", "nvidia/llama-nemotron-rerank-vl-1b-v2")

if NVIDIA_BASE_URL:
    print(f"Using self-hosted NIM at {NVIDIA_BASE_URL}")
else:
    print("Using NVIDIA-hosted endpoints (build.nvidia.com)")
if not all((NVIDIA_BASE_URL, EMBED_BASE_URL, RERANK_BASE_URL)) and not os.environ.get(
    "NVIDIA_API_KEY"
):
    os.environ["NVIDIA_API_KEY"] = getpass.getpass("NVIDIA API key (nvapi-...): ")


def nim_kwargs(base_url):
    """Keyword arguments for an NVIDIA client; ``None`` means hosted."""
    return {"base_url": base_url} if base_url else {}


CHAT_KWARGS = nim_kwargs(NVIDIA_BASE_URL)
EMBED_KWARGS = nim_kwargs(EMBED_BASE_URL)
RERANK_KWARGS = nim_kwargs(RERANK_BASE_URL)
print(f"Chat: {CHAT_MODEL} | Embed: {EMBED_MODEL} | Rerank: {RERANK_MODEL}")

### Tracing (optional)

To trace runs in [LangSmith](https://docs.langchain.com/langsmith/home), set these before running the agent:

```python
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = "<your-api-key>"
```

## Chat model

The graders below use structured output, so pick a model that supports tool calling. You can list the
tool-capable models available on your endpoint:

In [ ]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

[m.id for m in ChatNVIDIA.get_available_models(**CHAT_KWARGS) if m.supports_tools]

In [ ]:
llm = ChatNVIDIA(model=CHAT_MODEL, temperature=0, **CHAT_KWARGS)

## Index

Load a few blog posts, embed them into an in-memory vector store, and rerank the top matches.

In [ ]:
os.environ.setdefault("USER_AGENT", "langchain-nvidia-cookbook")

from langchain_community.document_loaders import WebBaseLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings, NVIDIARerank
from langchain_text_splitters import RecursiveCharacterTextSplitter

urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
]

docs_list = [doc for url in urls for doc in WebBaseLoader(url).load()]
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
doc_splits = text_splitter.split_documents(docs_list)

embeddings = NVIDIAEmbeddings(model=EMBED_MODEL, truncate="END", **EMBED_KWARGS)
vectorstore = InMemoryVectorStore.from_documents(doc_splits, embedding=embeddings)
reranker = NVIDIARerank(model=RERANK_MODEL, top_n=3, **RERANK_KWARGS)


def retrieve_docs(question):
    """Vector search for candidates, then keep the top reranked documents."""
    candidates = vectorstore.similarity_search(question, k=8)
    return list(reranker.compress_documents(documents=candidates, query=question))

## Components

Test structured output across the components.

In [ ]:
### Router
from typing import Literal

from langchain_core.messages import HumanMessage, SystemMessage
from pydantic import BaseModel, Field


class RouteQuery(BaseModel):
    """Route a user query to the most relevant datasource."""

    datasource: Literal["vectorstore", "websearch"] = Field(
        ...,
        description="Given a user question choose to route it to web search or a vectorstore.",
    )


structured_llm_router = llm.with_structured_output(RouteQuery)

router_instructions = """You are an expert at routing a user question to a vectorstore or web search.

The vectorstore contains documents related to agents, prompt engineering, and adversarial attacks.

Use the vectorstore for questions on these topics. For all else, and especially for current events, use web-search.

Return structured output with single key, datasource, that is 'websearch' or 'vectorstore' depending on the question."""

for q in [
    "Who is favored to win the next FIFA World Cup?",
    "What are the latest NVIDIA Nemotron models?",
    "What are the types of agent memory?",
]:
    print(q, "->", structured_llm_router.invoke(
        [SystemMessage(content=router_instructions), HumanMessage(content=q)]
    ))

In [ ]:
### Retrieval grader


class GradeDocuments(BaseModel):
    """Binary score for relevance check on retrieved documents."""

    binary_score: str = Field(
        description="Documents are relevant to the question, 'yes' or 'no'"
    )


structured_llm_grader = llm.with_structured_output(GradeDocuments)

doc_grader_instructions = """You are a grader assessing relevance of a retrieved document to a user question.

If the document contains keyword(s) or semantic meaning related to the question, grade it as relevant."""

doc_grader_prompt = """Here is the retrieved document: \n\n {document} \n\n Here is the user question: \n\n {question}.

Carefully and objectively assess whether the document contains at least some information that is relevant to the question.

Return structured output with single key, binary_score, that is 'yes' or 'no' score to indicate whether the document contains at least some information that is relevant to the question."""

question = "What is Chain of thought prompting?"
docs = retrieve_docs(question)
doc_grader_prompt_formatted = doc_grader_prompt.format(
    document=docs[0].page_content, question=question
)
structured_llm_grader.invoke(
    [SystemMessage(content=doc_grader_instructions)]
    + [HumanMessage(content=doc_grader_prompt_formatted)]
)

In [ ]:
### Generate

rag_prompt = """You are an assistant for question-answering tasks.

Here is the context to use to answer the question:

{context}

Think carefully about the above context.

Now, review the user question:

{question}

Provide an answer to this question using only the above context.

Use three sentences maximum and keep the answer concise.

Answer:"""


def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


docs_txt = format_docs(docs)
rag_prompt_formatted = rag_prompt.format(context=docs_txt, question=question)
generation = llm.invoke([HumanMessage(content=rag_prompt_formatted)])
print(generation.content)

In [ ]:
### Hallucination grader


class GradeHallucinations(BaseModel):
    """Binary score for hallucination present in generation answer."""

    binary_score: str = Field(
        description="Answer is grounded in the facts, 'yes' or 'no'"
    )


hallucination_grader = llm.with_structured_output(GradeHallucinations)

hallucination_grader_instructions = """You are a teacher grading a quiz.

You will be given FACTS and a STUDENT ANSWER.

Here is the grade criteria to follow:

(1) Ensure the STUDENT ANSWER is grounded in the FACTS.

(2) Ensure the STUDENT ANSWER does not contain "hallucinated" information outside the scope of the FACTS.

Score:

A score of yes means that the student's answer meets all of the criteria. This is the highest (best) score.

A score of no means that the student's answer does not meet all of the criteria. This is the lowest possible score you can give."""

hallucination_grader_prompt = """FACTS: \n\n {documents} \n\n STUDENT ANSWER: {generation}.

Return structured output with a single key, binary_score, that is 'yes' or 'no' to indicate whether the STUDENT ANSWER is grounded in the FACTS."""

hallucination_grader.invoke(
    [SystemMessage(content=hallucination_grader_instructions)]
    + [
        HumanMessage(
            content=hallucination_grader_prompt.format(
                documents=docs_txt, generation=generation.content
            )
        )
    ]
)

In [ ]:
### Answer grader


class GradeAnswer(BaseModel):
    """Binary score to assess answer addresses question."""

    binary_score: str = Field(
        description="Answer addresses the question, 'yes' or 'no'"
    )


answer_grader = llm.with_structured_output(GradeAnswer)

answer_grader_instructions = """You are a teacher grading a quiz.

You will be given a QUESTION and a STUDENT ANSWER.

Here is the grade criteria to follow:

(1) The STUDENT ANSWER helps to answer the QUESTION

Score:

A score of yes means that the student's answer meets all of the criteria. This is the highest (best) score.

The student can receive a score of yes if the answer contains extra information that is not explicitly asked for in the question.

A score of no means that the student's answer does not meet all of the criteria. This is the lowest possible score you can give."""

answer_grader_prompt = """QUESTION: \n\n {question} \n\n STUDENT ANSWER: {generation}.

Return structured output with a single key, binary_score, that is 'yes' or 'no' to indicate whether the STUDENT ANSWER meets the criteria."""

answer_grader.invoke(
    [SystemMessage(content=answer_grader_instructions)]
    + [
        HumanMessage(
            content=answer_grader_prompt.format(
                question="What are the types of agent memory?",
                generation="Agents use short-term memory (in-context learning) and long-term memory (an external vector store they can query).",
            )
        )
    ]
)

## Web search

With `TAVILY_API_KEY` set, the fallback path uses [Tavily](https://tavily.com/). Without it, an offline stub
keeps the notebook runnable with only an NVIDIA endpoint.

In [ ]:
if os.environ.get("TAVILY_API_KEY"):
    from langchain_tavily import TavilySearch

    tavily = TavilySearch(max_results=3)

    def search_web(query):
        response = tavily.invoke({"query": query})
        results = response.get("results", []) if isinstance(response, dict) else []
        return "\n".join(r["content"] for r in results)

else:

    def search_web(query):
        return f"(Web search is disabled; set TAVILY_API_KEY to search for: {query})"

## Graph

In [ ]:
### State
import operator
from typing import Annotated, List

from langchain_core.documents import Document
from typing_extensions import TypedDict


class GraphState(TypedDict):
    """Information propagated to, and modified in, each graph node."""

    question: str  # User question
    generation: str  # LLM generation
    web_search: str  # Binary decision to run web search
    max_retries: int  # Max number of retries for answer generation
    loop_step: Annotated[int, operator.add]  # Number of generations so far
    documents: List[Document]  # Retrieved documents

We'll implement these as a control flow in LangGraph.

In [ ]:
from langgraph.graph import END, START, StateGraph


### Nodes
def retrieve(state):
    """Retrieve and rerank documents from the vector store."""
    print("---RETRIEVE---")
    return {"documents": retrieve_docs(state["question"])}


def generate(state):
    """Generate an answer using RAG on the retrieved documents."""
    print("---GENERATE---")
    rag_prompt_formatted = rag_prompt.format(
        context=format_docs(state["documents"]), question=state["question"]
    )
    generation = llm.invoke([HumanMessage(content=rag_prompt_formatted)])
    return {"generation": generation.content, "loop_step": 1}


def grade_documents(state):
    """Keep relevant documents; request web search if any document is irrelevant."""
    print("---CHECK DOCUMENT RELEVANCE TO QUESTION---")
    filtered_docs = []
    web_search = "No"
    for doc in state["documents"]:
        result = structured_llm_grader.invoke(
            [SystemMessage(content=doc_grader_instructions)]
            + [
                HumanMessage(
                    content=doc_grader_prompt.format(
                        document=doc.page_content, question=state["question"]
                    )
                )
            ]
        )
        if result.binary_score.lower() == "yes":
            print("---GRADE: DOCUMENT RELEVANT---")
            filtered_docs.append(doc)
        else:
            print("---GRADE: DOCUMENT NOT RELEVANT---")
            web_search = "Yes"
    return {"documents": filtered_docs, "web_search": web_search}


def web_search(state):
    """Append web search results to the documents."""
    print("---WEB SEARCH---")
    documents = state.get("documents", [])
    documents.append(Document(page_content=search_web(state["question"])))
    return {"documents": documents}


### Edges
def route_question(state):
    """Route the question to web search or RAG."""
    print("---ROUTE QUESTION---")
    route = structured_llm_router.invoke(
        [SystemMessage(content=router_instructions)]
        + [HumanMessage(content=state["question"])]
    )
    if route.datasource == "websearch":
        print("---ROUTE QUESTION TO WEB SEARCH---")
        return "websearch"
    print("---ROUTE QUESTION TO RAG---")
    return "vectorstore"


def decide_to_generate(state):
    """Generate an answer, or add web search results first."""
    print("---ASSESS GRADED DOCUMENTS---")
    if state["web_search"] == "Yes":
        print("---DECISION: NOT ALL DOCUMENTS ARE RELEVANT TO QUESTION, INCLUDE WEB SEARCH---")
        return "websearch"
    print("---DECISION: GENERATE---")
    return "generate"


def grade_generation_v_documents_and_question(state):
    """Check that the generation is grounded in the documents and answers the question."""
    print("---CHECK HALLUCINATIONS---")
    max_retries = state.get("max_retries", 3)
    grounded = hallucination_grader.invoke(
        [SystemMessage(content=hallucination_grader_instructions)]
        + [
            HumanMessage(
                content=hallucination_grader_prompt.format(
                    documents=format_docs(state["documents"]),
                    generation=state["generation"],
                )
            )
        ]
    )
    if grounded.binary_score.lower() == "yes":
        print("---DECISION: GENERATION IS GROUNDED IN DOCUMENTS---")
        print("---GRADE GENERATION vs QUESTION---")
        answered = answer_grader.invoke(
            [SystemMessage(content=answer_grader_instructions)]
            + [
                HumanMessage(
                    content=answer_grader_prompt.format(
                        question=state["question"], generation=state["generation"]
                    )
                )
            ]
        )
        if answered.binary_score.lower() == "yes":
            print("---DECISION: GENERATION ADDRESSES QUESTION---")
            return "useful"
        if state["loop_step"] <= max_retries:
            print("---DECISION: GENERATION DOES NOT ADDRESS QUESTION---")
            return "not useful"
        print("---DECISION: MAX RETRIES REACHED---")
        return "max retries"
    if state["loop_step"] <= max_retries:
        print("---DECISION: GENERATION IS NOT GROUNDED IN DOCUMENTS, RE-TRY---")
        return "not supported"
    print("---DECISION: MAX RETRIES REACHED---")
    return "max retries"

### Graph build

In [ ]:
workflow = StateGraph(GraphState)

workflow.add_node("websearch", web_search)
workflow.add_node("retrieve", retrieve)
workflow.add_node("grade_documents", grade_documents)
workflow.add_node("generate", generate)

workflow.add_conditional_edges(
    START,
    route_question,
    {"websearch": "websearch", "vectorstore": "retrieve"},
)
workflow.add_edge("websearch", "generate")
workflow.add_edge("retrieve", "grade_documents")
workflow.add_conditional_edges(
    "grade_documents",
    decide_to_generate,
    {"websearch": "websearch", "generate": "generate"},
)
workflow.add_conditional_edges(
    "generate",
    grade_generation_v_documents_and_question,
    {
        "not supported": "generate",
        "useful": END,
        "not useful": "websearch",
        "max retries": END,
    },
)

graph = workflow.compile()

try:
    from IPython.display import Image, display

    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    # Rendering the PNG needs network access to mermaid.ink.
    print(graph.get_graph().draw_mermaid())

In [ ]:
# A question the vector store can answer
inputs = {"question": "What are the types of agent memory?", "max_retries": 3}
for event in graph.stream(inputs, stream_mode="values"):
    print(event.get("generation", ""))

In [ ]:
# A current-events question that is routed to web search
inputs = {"question": "What are the latest NVIDIA Nemotron models?", "max_retries": 3}
for event in graph.stream(inputs, stream_mode="values"):
    print(event.get("generation", ""))

## Related

- [Cookbook index](https://github.com/langchain-ai/langchain-nvidia/tree/main/cookbook)
- [`langchain-nvidia-ai-endpoints` README](https://github.com/langchain-ai/langchain-nvidia/blob/main/libs/ai-endpoints/README.md)
- [Model registry: current and deprecated model IDs](https://github.com/langchain-ai/langchain-nvidia/blob/main/libs/ai-endpoints/docs/model_registry.md)
- [NVIDIA NIM for LLMs documentation](https://docs.nvidia.com/nim/large-language-models/latest/introduction.html)
- [NeMo Retriever embedding NIM](https://docs.nvidia.com/nim/nemo-retriever/text-embedding/latest/overview.html)
- [NeMo Retriever reranking NIM](https://docs.nvidia.com/nim/nemo-retriever/text-reranking/latest/overview.html)